In [1]:
from statsmodels.tsa.stattools import adfuller

In [6]:
import pandas as pd

df = pd.read_csv("../data/processed/market_data.csv")

df["Date"] = pd.to_datetime(df["Date"])

df = df.sort_values(["Ticker", "Date"])

df.head()

,Date,Close,High,Low,Open,Volume,Ticker
2888,2015-01-02,59.205643,59.234297,59.040882,59.048047,2218800,BND
2889,2015-01-05,59.377594,59.399083,59.241487,59.270142,5820100,BND
2890,2015-01-06,59.549515,59.728600,59.477882,59.477882,3887600,BND
2891,2015-01-07,59.585320,59.656953,59.492197,59.556665,2433400,BND
2892,2015-01-08,59.492184,59.535162,59.434875,59.535162,1873400,BND


In [7]:
df["Daily_Return"] = df.groupby("Ticker")["Close"].pct_change()

In [8]:
def adf_test(series, name):
    result = adfuller(series.dropna())

    print("\n" + "="*50)
    print(f"ADF TEST: {name}")
    print("="*50)

    print(f"ADF Statistic: {result[0]}")
    print(f"p-value: {result[1]}")

    for key, value in result[4].items():
        print(f"Critical Value {key}: {value}")

    if result[1] <= 0.05:
        print("👉 RESULT: Stationary ✅")
    else:
        print("👉 RESULT: Non-Stationary ❌")

In [9]:
for ticker in ["TSLA", "SPY", "BND"]:
    series = df[df["Ticker"] == ticker]["Close"]
    adf_test(series, f"{ticker} - Close Price")


ADF TEST: TSLA - Close Price
ADF Statistic: -1.0695991394177442
p-value: 0.7270419411549228
Critical Value 1%: -3.432638524105552
Critical Value 5%: -2.8625511137463935
Critical Value 10%: -2.567308245513228
👉 RESULT: Non-Stationary ❌

ADF TEST: SPY - Close Price
ADF Statistic: 1.308943580149355
p-value: 0.996658545157695
Critical Value 1%: -3.4326241980933294
Critical Value 5%: -2.8625447866555227
Critical Value 10%: -2.567304877006608
👉 RESULT: Non-Stationary ❌

ADF TEST: BND - Close Price
ADF Statistic: -1.0478698984113661
p-value: 0.7354071572882626
Critical Value 1%: -3.432627366129768
Critical Value 5%: -2.862546185823105
Critical Value 10%: -2.567305621914905
👉 RESULT: Non-Stationary ❌


 ADF TEST INTERPRETATION (Closing Prices)
 1. TSLA (Tesla)
ADF Statistic: -1.07
p-value: 0.727
Interpretation:
p-value is much greater than 0.05
We fail to reject the null hypothesis
TSLA closing prices are non-stationary
 Meaning:

TSLA has a strong trend over time
Its mean and variance change over time
This is typical for high-growth stocks
2. SPY (S&P 500 ETF)
ADF Statistic: 1.31
p-value: 0.996
Interpretation:
Extremely high p-value → very strong non-stationarity
SPY follows a clear long-term upward trend

Meaning:

Broad market index grows over time
Strong trend = non-stationary series
Not suitable for ARIMA without transformation
3. BND (Bond ETF)
ADF Statistic: -1.05
p-value: 0.735
Interpretation:
Also non-stationary
Even stable bond ETFs show gradual trends over time
 Meaning:

Interest rate changes and bond pricing trends affect long-term behavior
Still not stationary

## Stationarity Test (ADF Results)

The Augmented Dickey-Fuller (ADF) test was applied to the closing prices of TSLA, SPY, and BND to assess stationarity.

The results show that all three assets have p-values significantly greater than 0.05, indicating failure to reject the null hypothesis of non-stationarity. This confirms that the price series contain trends and their statistical properties, such as mean and variance, change over time.

TSLA exhibits non-stationarity due to strong growth and volatility characteristics. SPY shows a strong upward trend reflecting long-term market growth. BND also demonstrates non-stationarity due to macroeconomic and interest rate effects.

These results imply that raw price data is not suitable for ARIMA modeling. Therefore, differencing must be applied to transform the series into a stationary form before forecasting.

In [10]:
for ticker in ["TSLA", "SPY", "BND"]:
    series = df[df["Ticker"] == ticker]["Daily_Return"]
    adf_test(series, f"{ticker} - Daily Returns")


ADF TEST: TSLA - Daily Returns
ADF Statistic: -53.97189280425144
p-value: 0.0
Critical Value 1%: -3.4326178883893763
Critical Value 5%: -2.862541999961805
Critical Value 10%: -2.567303393388591
👉 RESULT: Stationary ✅

ADF TEST: SPY - Daily Returns
ADF Statistic: -17.572181203923254
p-value: 4.046307933439359e-30
Critical Value 1%: -3.4326241980933294
Critical Value 5%: -2.8625447866555227
Critical Value 10%: -2.567304877006608
👉 RESULT: Stationary ✅

ADF TEST: BND - Daily Returns
ADF Statistic: -15.221188310904532
p-value: 5.482342788443661e-28
Critical Value 1%: -3.432627366129768
Critical Value 5%: -2.862546185823105
Critical Value 10%: -2.567305621914905
👉 RESULT: Stationary ✅


ADF TEST INTERPRETATION (Daily Returns)
1. TSLA (Tesla)
ADF Statistic: -53.97
p-value: 0.0
Interpretation:
p-value < 0.05 → reject null hypothesis
The series is strongly stationary

 Meaning:

TSLA returns fluctuate around a stable mean
No long-term trend in returns
Suitable for volatility modeling
2. SPY (S&P 500 ETF)
ADF Statistic: -17.57
p-value: ~ 4e-30
Interpretation:
Strong evidence of stationarity
Returns behave consistently over time

 Meaning:

Market returns are mean-reverting
Suitable for forecasting risk and portfolio behavior
3. BND (Bond ETF)
ADF Statistic: -15.22
p-value: ~ 5e-28
Interpretation:
Strongly stationary series
Very stable return behavior

 Meaning:

Bond returns are low volatility and predictable in distribution
Ideal for risk reduction in portfolios
## Stationarity Test (Daily Returns)

The Augmented Dickey-Fuller (ADF) test was applied to the daily returns of TSLA, SPY, and BND. The results show that all return series are strongly stationary, with p-values effectively equal to zero.

This indicates that daily returns fluctuate around a constant mean and do not exhibit long-term trends. Unlike price series, return series have stable statistical properties over time, making them suitable for time series modeling and volatility analysis.

The stationarity of returns confirms their suitability for financial modeling tasks such as risk estimation, portfolio optimization, and volatility forecasting. This contrasts with raw price data, which was previously found to be non-stationary and therefore unsuitable for direct ARIMA modeling without transformation.

In [11]:
tsla = df[df["Ticker"] == "TSLA"].copy()

tsla = tsla.sort_values("Date")

tsla["Daily_Return"] = tsla["Close"].pct_change()

tsla = tsla.dropna()

In [12]:
var_95 = tsla["Daily_Return"].quantile(0.05)

var_99 = tsla["Daily_Return"].quantile(0.01)

print("VaR 95%:", var_95)
print("VaR 99%:", var_99)

VaR 95%: -0.05166375515284548
VaR 99%: -0.08990644871169305


In [13]:
mean_return = tsla["Daily_Return"].mean()
std_return = tsla["Daily_Return"].std()

sharpe_ratio = mean_return / std_return

print("Sharpe Ratio:", sharpe_ratio)

Sharpe Ratio: 0.050044827310504


 Value at Risk (VaR) Interpretation
 VaR 95% = -0.0517

 Meaning:
On a “normal bad day”, TSLA can lose up to 5.17%
95% of the time, losses are better than this
 VaR 99% = -0.0899

Meaning:

In extreme conditions, TSLA can lose up to 8.99% in a single day
These are rare but severe market shocks
Financial Insight
TSLA has significant downside risk
Tail risk (extreme loss risk) is high
This is typical for high-growth tech stocks

 Conclusion:

TSLA is highly exposed to extreme market movements, especially during crisis periods.

2. Sharpe Ratio Interpretation
 Sharpe = 0.05

Meaning:

Very low risk-adjusted return
Returns are only slightly higher than risk taken

## Risk Metrics Analysis (TSLA)

The Value at Risk (VaR) analysis indicates that TSLA has a 5% daily loss threshold of approximately -5.17%, and a 1% extreme loss threshold of approximately -8.99%. This suggests that the asset is exposed to significant downside risk, particularly during market stress conditions.

The Sharpe Ratio of 0.05 indicates very low risk-adjusted returns, meaning that the returns generated by TSLA are not sufficient to compensate for its high volatility.

Overall, TSLA exhibits a high-risk, high-volatility profile with limited efficiency in risk-adjusted performance. While it offers potential for strong gains, the magnitude of its fluctuations reduces its attractiveness from a risk-adjusted investment perspective.